# System 2: Pre-Evaluation Baseline Run (Gold Standard v2)

**Purpose:** Run System 2 (with Tier-1 features: Reflexion verifier, Few-Shot prompt, `search_section` reranker, `sub_query`) against the **v2 gold standard**, a rewritten version of the 77-question ablation set in which every query contains the entity (ticker OR company name) and fiscal year explicitly. This replaces the v1 CSV that was methodologically flawed (see EVAL_DECISION_LOG).

## Reference: System 1 (Monolith) best-config scores on the ORIGINAL v1 CSV

From `configs/best_config.yaml` (Optuna, 50 trials, best_trial=27). **Caveat:** these numbers were obtained on the underspecified v1 CSV and are NOT directly comparable to System 2 on v2. A System 1 re-run on v2 is a separate follow-up task.

| Metric | System 1 (v1) |
|---|---|
| Context Precision | 0.4392 |
| Context Recall | 0.2625 |
| Faithfulness | 0.9722 |
| **Composite** | **0.5580** |

## Gold Standard v2 design

- All 77 questions rewritten in **natural German sentence form**
- Stratified **50/50 ticker form / name form** within each query type (38 ticker, 39 name total)
- Stratification is deterministic (`random.Random(42)` in `scripts/build_gold_standard_v2.py`)
- Entity name mapping: AAPL→Apple, MSFT→Microsoft, AMZN→Amazon, GOOGL→Alphabet
- Each row carries a new `entity_form` column (`ticker` or `name`) for sub-aggregation

## Methodological notes

- **Preliminary dev-run.** Optuna HPs inherited unchanged from the v1-tuned `best_config.yaml` (user decision: HPs remain frozen). Results are diagnostic, not to be fed back into System 2 tuning.
- **Judge model:** `gemini-2.0-flash` (same as System 1 evaluation, documented in `EVAL_DECISION_LOG.md`).
- **Reflection + Few-Shot are enabled by default.**
- The entity-form split also tests **surface-form robustness** — are scores comparable between ticker-phrased and name-phrased queries?

In [1]:
import json
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# Resolve project root regardless of where the notebook runs from
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
# Quiet noisy libraries
for noisy in ["httpx", "urllib3", "chromadb", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore")

from src.common.ingestion import ProcessedFiling
from src.systems.rag_agent.pipeline import AgentRAGPipeline
from src.evaluation.gold_standard_loader import load_gold_standard
from src.evaluation.ragas_evaluator import evaluate_run

print("Imports done.")

Project root: /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis


11:58:26 [INFO] numexpr.utils: NumExpr defaulting to 14 threads.


Imports done.


## 1. Load gold standard and filings

In [2]:
GOLD_CSV = PROJECT_ROOT / "notebooks" / "experiments" / "sys1_rag_monolith" / "ablation_test_data_v2.csv"
gold_items = load_gold_standard(GOLD_CSV)
print(f"Loaded {len(gold_items)} gold-standard items from {GOLD_CSV.name}")

# Breakdown by query type
from collections import Counter
type_counts = Counter(item.query_type for item in gold_items)
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Entity-form distribution (v2 feature)
form_counts = Counter(item.entity_form for item in gold_items)
print("\nEntity-form distribution:")
for form, c in sorted(form_counts.items(), key=lambda kv: (kv[0] is None, kv[0])):
    print(f"  {form}: {c}")

11:58:38 [INFO] src.evaluation.gold_standard_loader: Loaded 77 gold-standard items from ablation_test_data_v2.csv (filter=None)


Loaded 77 gold-standard items from ablation_test_data_v2.csv
  Cross-Sec: 20
  Multi-Comp: 20
  Multi-Year: 17
  Single: 20

Entity-form distribution:
  name: 39
  ticker: 38


In [3]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

filings = []
for meta_file in DATA_DIR.rglob("*.meta.json"):
    md_file = meta_file.with_suffix("").with_suffix(".md")
    if md_file.exists():
        filings.append(ProcessedFiling.from_files(md_file, meta_file))

print(f"Loaded {len(filings)} filings from {DATA_DIR}")
for f in filings:
    fy = f.metadata.fiscal_year_end[:4] if f.metadata.fiscal_year_end else "?"
    print(f"  {f.metadata.ticker} FY{fy}")

Loaded 12 filings from /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/data/processed
  AMZN FY2023
  AMZN FY2022
  AMZN FY2024
  AAPL FY2022
  AAPL FY2023
  AAPL FY2024
  GOOGL FY2024
  GOOGL FY2022
  GOOGL FY2023
  MSFT FY2023
  MSFT FY2022
  MSFT FY2024


## 2. Build System 2 pipeline with Tier-1 defaults

Reflection is enabled (default), sub_query-capable `search_section` with FlashRank reranker is active, and the Few-Shot examples are embedded in the system prompt. Retrieval HPs are inherited unchanged from `configs/best_config.yaml`.

In [4]:
print("Building AgentRAGPipeline (this may take a moment for vectorstore init)...")
pipeline = AgentRAGPipeline()
pipeline.build(filings)
print("Pipeline built.")
print(f"Params: {pipeline.params}")

11:58:38 [INFO] src.systems.rag_agent.pipeline: Building agent pipeline: chunk_size=1000, overlap=10%, bm25_weight=0.50, pre_k=15
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AMZN FY2023: 412 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AMZN FY2022: 814 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AMZN FY2024: 1215 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AAPL FY2022: 1519 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AAPL FY2023: 1804 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked AAPL FY2024: 2092 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked GOOGL FY2024: 2324 chunks (chunk_size=1000, overlap=100)
11:58:38 [INFO] src.systems.rag_monolith.chunker: Chunked GOOGL FY2022: 2

Building AgentRAGPipeline (this may take a moment for vectorstore init)...


11:58:38 [INFO] src.systems.rag_monolith.chunker: Total: 3909 documents from 12 filings (chunk_size=1000, overlap_pct=10%)
/Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/src/common/llm_client.py:119: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  inner = VertexAIEmbeddings(
11:58:38 [INFO] src.common.llm_client: Embeddings initialized: model=gemini-embedding-001, project=master-thesis-489320, location=global
11:58:38 [INFO] src.common.retrieval: Building vectorstore: 3909 docs → data/vectorstores/rag_monolith/cs1000_ov10/sec_10k_filings
12:00:53 [INFO] src.common.retrieval: Vectorstore built: 3909 documents indexed
12:00:53 [INFO] src.c

Pipeline built.
Params: {'chunk_size': 1000, 'chunk_overlap_pct': 0.1, 'bm25_weight': 0.5, 'pre_rerank_top_k': 15, 'post_rerank_top_k': 4, 'max_iterations': 5, 'reflection_enabled': True}


## 3. Run all 77 queries

Per-query error handling (a single failure does not abort the run). Intermediate results are persisted to JSON after every query so a mid-run crash does not lose progress. Expected wall-clock time: ~30-60 min. Expected cost: ~$1-3.

In [5]:
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
intermediate_path = RESULTS_DIR / f"sys2_baseline_raw_{run_timestamp}.json"
print(f"Intermediate results will be saved to: {intermediate_path}")


def coerce_answer_to_str(answer) -> str:
    """Gemini 2.5 may return [{'type':'text','text':...}] lists. Coerce to str."""
    if isinstance(answer, str):
        return answer
    if isinstance(answer, list):
        return "\n".join(
            p.get("text", str(p)) if isinstance(p, dict) else str(p)
            for p in answer
        )
    return str(answer)


raw_results = []
total_start = time.perf_counter()

for i, item in enumerate(gold_items, 1):
    print(f"[{i:3d}/{len(gold_items)}] {item.query_type:10s} | {item.doc_refs:25s} | {item.question[:70]}")
    try:
        res = pipeline.query(item.question)
        answer_str = coerce_answer_to_str(res.answer)
        entry = {
            "id": item.id,
            "question": item.question,
            "ground_truth": item.ground_truth,
            "query_type": item.query_type,
            "doc_refs": item.doc_refs,
            "entity_form": item.entity_form,
            "answer": answer_str,
            "contexts": res.contexts,
            "tool_calls": [tc["tool"] for tc in res.tool_calls_log],
            "num_steps": res.metrics.num_steps,
            "latency_seconds": res.metrics.latency_seconds,
            "prompt_tokens": res.metrics.token_usage.prompt_tokens,
            "completion_tokens": res.metrics.token_usage.completion_tokens,
            "total_tokens": res.metrics.token_usage.total_tokens,
            "estimated_cost_usd": res.metrics.estimated_cost_usd,
            "was_revised": res.was_revised,
            "reflection_status": (
                res.reflection_verdict.status if res.reflection_verdict else None
            ),
            "reflection_issues": (
                res.reflection_verdict.issues if res.reflection_verdict else []
            ),
            "error": None,
        }
        print(
            f"       -> {res.metrics.latency_seconds:.1f}s | "
            f"{res.metrics.num_steps} tool calls | "
            f"{res.metrics.token_usage.total_tokens} tok | "
            f"revised={res.was_revised}"
        )
    except Exception as e:
        print(f"       !! ERROR: {e}")
        entry = {
            "id": item.id,
            "question": item.question,
            "ground_truth": item.ground_truth,
            "query_type": item.query_type,
            "doc_refs": item.doc_refs,
            "entity_form": item.entity_form,
            "answer": "",
            "contexts": [],
            "tool_calls": [],
            "num_steps": 0,
            "latency_seconds": 0.0,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": 0.0,
            "was_revised": False,
            "reflection_status": None,
            "reflection_issues": [],
            "error": str(e),
        }

    raw_results.append(entry)

    # Persist after each query (crash-safe)
    with open(intermediate_path, "w", encoding="utf-8") as f:
        json.dump(raw_results, f, ensure_ascii=False, indent=2)

total_elapsed = time.perf_counter() - total_start
print(f"\nAll queries done in {total_elapsed/60:.1f} minutes. Saved to {intermediate_path}")

Intermediate results will be saved to: /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/data/results/sys2_baseline_raw_20260419_120053.json
[  1/77] Single     | AAPL_2024                 | Wie hoch war der Gesamtumsatz (Total Revenue) von Apple in FY2024?


12:00:55 [INFO] src.common.retrieval: FlashRank ranker loaded: ms-marco-MiniLM-L-12-v2
12:00:56 [INFO] src.systems.rag_agent.tools.search_section: search_section: AAPL FY2024 Financial Statements (sub_query='total revenue') → 4/15 chunks
12:00:59 [INFO] src.systems.rag_agent.pipeline: Reflection verdict: status=accept, issues=[]
12:00:59 [INFO] src.systems.rag_agent.pipeline: Agent query completed: 1 tool calls, 6.06s, 5422 tokens (reflection=on, revised=False)


       -> 6.1s | 1 tool calls | 5422 tok | revised=False
[  2/77] Single     | AAPL_2024                 | Wie hoch war das Net Income von Apple in FY2024?


12:01:00 [INFO] src.systems.rag_agent.tools.search_section: search_section: AAPL FY2024 Financial Statements (sub_query='Net Income') → 4/15 chunks
12:01:05 [INFO] src.systems.rag_agent.pipeline: Reflection verdict: status=accept, issues=[]
12:01:05 [INFO] src.systems.rag_agent.pipeline: Agent query completed: 1 tool calls, 6.30s, 5573 tokens (reflection=on, revised=False)


       -> 6.3s | 1 tool calls | 5573 tok | revised=False
[  3/77] Single     | AAPL_2024                 | Wie hoch war die Debt-to-Equity Ratio von Apple am Ende von FY2024?


...

       -> 23.5s | 10 tool calls | 63922 tok | revised=True
[ 76/77] Multi-Comp | All_4_2024                | Welches der 4 Unternehmen (Apple, Microsoft, Amazon, Alphabet) hatte d


12:18:29 [INFO] src.systems.rag_agent.tools.search_section: search_section: AAPL FY2024 Financial Statements (sub_query='total net sales fiscal 2024') → 4/15 chunks
12:18:29 [INFO] src.systems.rag_agent.tools.search_section: search_section: MSFT FY2024 Financial Statements (sub_query='total revenue fiscal 2024') → 2/2 chunks
12:18:30 [INFO] src.systems.rag_agent.tools.search_section: search_section: AMZN FY2024 Financial Statements (sub_query='total net sales fiscal 2024') → 4/15 chunks
12:18:30 [INFO] src.systems.rag_agent.tools.search_section: search_section: GOOGL FY2024 Financial Statements (sub_query='total revenues fiscal 2024') → 4/15 chunks
12:18:32 [INFO] src.systems.rag_agent.tools.search_section: search_section: MSFT FY2024 Financial Statements (sub_query='total revenue') → 2/2 chunks
12:18:32 [INFO] src.systems.rag_agent.tools.search_section: search_section: AMZN FY2024 Financial Statements (sub_query='total net sales') → 4/15 chunks
12:18:34 [INFO] src.systems.rag_agent.to

       -> 16.3s | 9 tool calls | 34484 tok | revised=False
[ 77/77] Multi-Comp | All_4_2024                | Welches der 4 Unternehmen (AAPL, MSFT, AMZN, GOOGL) hat die höchste De


12:18:46 [INFO] src.systems.rag_agent.tools.search_section: search_section: MSFT FY2024 Financial Statements (sub_query='total debt and total equity') → 2/2 chunks
12:18:46 [INFO] src.systems.rag_agent.tools.search_section: search_section: AAPL FY2024 Financial Statements (sub_query='total debt and total equity') → 4/15 chunks
12:18:46 [INFO] src.systems.rag_agent.tools.search_section: search_section: AMZN FY2024 Financial Statements (sub_query='total debt and total equity') → 4/15 chunks
12:18:46 [INFO] src.systems.rag_agent.tools.search_section: search_section: GOOGL FY2024 Financial Statements (sub_query='total debt and total equity') → 4/15 chunks
12:18:55 [INFO] src.systems.rag_agent.tools.search_section: search_section: MSFT FY2024 Financial Statements (sub_query="total liabilities and total stockholders' equity") → 2/2 chunks
12:18:58 [INFO] src.systems.rag_agent.tools.calculate: calculate: '96662 / 56950' → 1.69731
12:18:58 [INFO] src.systems.rag_agent.tools.calculate: calculat

       -> 23.1s | 8 tool calls | 25587 tok | revised=False

All queries done in 18.2 minutes. Saved to /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/data/results/sys2_baseline_raw_20260419_120053.json


## 4. Aggregate efficiency metrics

In [6]:
ok_rows = [r for r in raw_results if r["error"] is None]
err_rows = [r for r in raw_results if r["error"] is not None]

n = len(ok_rows)
if n == 0:
    raise RuntimeError("All queries errored — cannot aggregate metrics.")

avg_latency = sum(r["latency_seconds"] for r in ok_rows) / n
avg_tokens = sum(r["total_tokens"] for r in ok_rows) / n
total_tokens = sum(r["total_tokens"] for r in ok_rows)
total_cost = sum(r["estimated_cost_usd"] for r in ok_rows)
avg_steps = sum(r["num_steps"] for r in ok_rows) / n
revision_rate = sum(1 for r in ok_rows if r["was_revised"]) / n

print("Efficiency metrics (System 2, Tier-1 defaults):")
print(f"  Successful queries:    {n} / {len(gold_items)}")
print(f"  Errored queries:       {len(err_rows)}")
print(f"  Avg latency:           {avg_latency:.2f} s")
print(f"  Avg tokens per query:  {avg_tokens:.0f}")
print(f"  Total tokens:          {total_tokens:,}")
print(f"  Avg tool calls:        {avg_steps:.2f}")
print(f"  Revision rate:         {revision_rate:.1%}")
print(f"  Estimated cost (USD):  ${total_cost:.4f}")

if err_rows:
    print("\nErrors:")
    for r in err_rows:
        print(f"  id={r['id']}: {r['error']}")

Efficiency metrics (System 2, Tier-1 defaults):
  Successful queries:    77 / 77
  Errored queries:       0
  Avg latency:           14.19 s
  Avg tokens per query:  18959
  Total tokens:          1,459,865
  Avg tool calls:        4.17
  Revision rate:         7.8%
  Estimated cost (USD):  $0.4733


## 5. RAGAS evaluation (same judge model as System 1)

In [7]:
# Only evaluate successful rows; errored ones would skew the aggregate
eval_items = [
    item for item, r in zip(gold_items, raw_results) if r["error"] is None
]
eval_answers = [r["answer"] for r in raw_results if r["error"] is None]
eval_contexts = [r["contexts"] for r in raw_results if r["error"] is None]

print(f"Running RAGAS on {len(eval_items)} successful queries...")
scores = evaluate_run(
    gold_standard=eval_items,
    answers=eval_answers,
    contexts=eval_contexts,
)

print("\nRAGAS scores:")
for k, v in scores.to_dict().items():
    print(f"  {k}: {v}")

12:19:06 [INFO] src.evaluation.ragas_evaluator: Running RAGAS evaluation on 77 samples...
12:19:06 [INFO] src.common.llm_client: LLM initialized: model=gemini-2.0-flash, temperature=0.0, project=master-thesis-489320, location=global
12:19:06 [INFO] src.common.llm_client: Embeddings initialized: model=gemini-embedding-001, project=master-thesis-489320, location=global


Running RAGAS on 77 successful queries...


Evaluating:   0%|          | 0/231 [00:00<?, ?it/s]

12:25:10 [INFO] src.evaluation.ragas_evaluator: RAGAS scores: precision=0.669, recall=0.481, faithfulness=0.768 → composite=0.639



RAGAS scores:
  context_precision: 0.6694
  context_recall: 0.4805
  faithfulness: 0.768
  composite_score: 0.6393


## 5b. RAGAS sub-aggregation by entity form (ticker vs name)

The v2 gold standard splits queries 50/50 into ticker form (`AAPL`, `MSFT`, ...) and name form (`Apple`, `Microsoft`, ...). Running RAGAS separately on each subset reveals whether System 2 behaves differently depending on the entity surface form — a direct robustness signal.

In [8]:
scores_by_form: dict[str, dict] = {}

for form in ("ticker", "name"):
    sub_items = [
        item for item, r in zip(gold_items, raw_results)
        if r["error"] is None and r.get("entity_form") == form
    ]
    sub_answers = [
        r["answer"] for r in raw_results
        if r["error"] is None and r.get("entity_form") == form
    ]
    sub_contexts = [
        r["contexts"] for r in raw_results
        if r["error"] is None and r.get("entity_form") == form
    ]

    if not sub_items:
        print(f"[{form}] no successful rows — skipping.")
        continue

    print(f"\n[{form}] Running RAGAS on {len(sub_items)} queries...")
    sub_scores = evaluate_run(
        gold_standard=sub_items,
        answers=sub_answers,
        contexts=sub_contexts,
    )
    scores_by_form[form] = sub_scores.to_dict()

    print(f"[{form}] scores:")
    for k, v in scores_by_form[form].items():
        print(f"  {k}: {v}")

# Summary comparison ticker vs name
print("\n" + "=" * 60)
print(f"{'Metric':<22} {'ticker':>12} {'name':>12} {'Delta':>12}")
print("-" * 60)
if "ticker" in scores_by_form and "name" in scores_by_form:
    for key in ["context_precision", "context_recall", "faithfulness", "composite_score"]:
        t = scores_by_form["ticker"][key]
        n = scores_by_form["name"][key]
        print(f"{key:<22} {t:>12.4f} {n:>12.4f} {(n-t):>+12.4f}")
    print("\nInterpretation: positive delta means System 2 performs BETTER with name form.")
    print("A large absolute delta in either direction = surface-form sensitivity.")

12:25:10 [INFO] src.evaluation.ragas_evaluator: Running RAGAS evaluation on 38 samples...
12:25:10 [INFO] src.common.llm_client: LLM initialized: model=gemini-2.0-flash, temperature=0.0, project=master-thesis-489320, location=global
12:25:10 [INFO] src.common.llm_client: Embeddings initialized: model=gemini-embedding-001, project=master-thesis-489320, location=global



[ticker] Running RAGAS on 38 queries...


Evaluating:   0%|          | 0/114 [00:00<?, ?it/s]

12:28:05 [INFO] src.evaluation.ragas_evaluator: RAGAS scores: precision=0.698, recall=0.474, faithfulness=0.750 → composite=0.641
12:28:05 [INFO] src.evaluation.ragas_evaluator: Running RAGAS evaluation on 39 samples...
12:28:05 [INFO] src.common.llm_client: LLM initialized: model=gemini-2.0-flash, temperature=0.0, project=master-thesis-489320, location=global
12:28:05 [INFO] src.common.llm_client: Embeddings initialized: model=gemini-embedding-001, project=master-thesis-489320, location=global


[ticker] scores:
  context_precision: 0.6978
  context_recall: 0.4737
  faithfulness: 0.7504
  composite_score: 0.6406

[name] Running RAGAS on 39 queries...


Evaluating:   0%|          | 0/117 [00:00<?, ?it/s]

12:31:13 [INFO] src.evaluation.ragas_evaluator: RAGAS scores: precision=0.670, recall=0.513, faithfulness=0.797 → composite=0.660


[name] scores:
  context_precision: 0.6696
  context_recall: 0.5128
  faithfulness: 0.7966
  composite_score: 0.6597

Metric                       ticker         name        Delta
------------------------------------------------------------
context_precision            0.6978       0.6696      -0.0282
context_recall               0.4737       0.5128      +0.0391
faithfulness                 0.7504       0.7966      +0.0462
composite_score              0.6406       0.6597      +0.0191

Interpretation: positive delta means System 2 performs BETTER with name form.
A large absolute delta in either direction = surface-form sensitivity.


## 6. Side-by-side comparison with System 1 best-config

In [9]:
SYS1_BEST = {
    "context_precision": 0.4392,
    "context_recall": 0.2625,
    "faithfulness": 0.9722,
    "composite_score": 0.5580,
}

sys2 = scores.to_dict()

print(f"{'Metric':<22} {'Sys 1 (tuned)':>14} {'Sys 2 (Tier-1)':>16} {'Delta':>10}")
print("-" * 64)
for key in ["context_precision", "context_recall", "faithfulness", "composite_score"]:
    s1 = SYS1_BEST[key]
    s2 = sys2[key]
    delta = s2 - s1
    arrow = "+" if delta >= 0 else ""
    print(f"{key:<22} {s1:>14.4f} {s2:>16.4f} {arrow}{delta:>9.4f}")

print("\nNotes:")
print("- Sys1 was Optuna-tuned on this exact CSV; Sys2 inherited the HPs unchanged.")
print("- Sys2 improvements beyond Sys1 on this CSV are a LOWER BOUND for the agentic advantage.")
print("- Results are purely diagnostic; not to be used for further Sys2 tuning.")

Metric                  Sys 1 (tuned)   Sys 2 (Tier-1)      Delta
----------------------------------------------------------------
context_precision              0.4392           0.6694 +   0.2302
context_recall                 0.2625           0.4805 +   0.2180
faithfulness                   0.9722           0.7680   -0.2042
composite_score                0.5580           0.6393 +   0.0813

Notes:
- Sys1 was Optuna-tuned on this exact CSV; Sys2 inherited the HPs unchanged.
- Sys2 improvements beyond Sys1 on this CSV are a LOWER BOUND for the agentic advantage.
- Results are purely diagnostic; not to be used for further Sys2 tuning.


## 7. Persist aggregated results

In [10]:
summary = {
    "run_timestamp": run_timestamp,
    "gold_csv": str(GOLD_CSV.relative_to(PROJECT_ROOT)),
    "num_queries": len(gold_items),
    "num_successful": len(ok_rows),
    "num_errored": len(err_rows),
    "pipeline_params": pipeline.params,
    "efficiency": {
        "avg_latency_seconds": round(avg_latency, 3),
        "avg_tokens_per_query": int(avg_tokens),
        "total_tokens": int(total_tokens),
        "avg_tool_calls": round(avg_steps, 2),
        "revision_rate": round(revision_rate, 4),
        "estimated_total_cost_usd": round(total_cost, 4),
    },
    "ragas_scores": scores.to_dict(),
    "ragas_scores_by_entity_form": scores_by_form,
    "sys1_reference_v1": SYS1_BEST,
    "disclaimer": (
        "Pre-evaluation dev run on ablation_test_data_v2.csv (natural-language queries "
        "with stratified ticker/name entity form). Sys1 reference scores are from the "
        "ORIGINAL v1 CSV and are not directly comparable without a Sys1 re-run on v2. "
        "Results are diagnostic and must not be used to further tune System 2."
    ),
}

summary_path = RESULTS_DIR / f"sys2_baseline_summary_{run_timestamp}.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Summary saved to: {summary_path}")
print(f"Raw per-query data: {intermediate_path}")

Summary saved to: /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/data/results/sys2_baseline_summary_20260419_120053.json
Raw per-query data: /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/data/results/sys2_baseline_raw_20260419_120053.json
